# Beyond RAG — GEMS-RAG pipeline (Colab)

Graph-grounded multimodal retrieval over the MUTCD 11th Edition, plus the
MUTCD-150 benchmark runner and ablation harness.

**Run order.** Sections 0–2 once per session. Section 3 onwards is optional.

**Before you start:**
1. Runtime -> Change runtime type -> **A100 GPU**
2. Put the MUTCD PDF in `Drive/MyDrive/MRAG/` (any `*.pdf` filename works)
3. Add API keys under the key icon in the left sidebar (Colab Secrets)

First run does a full ingest (~30–45 min on A100). Later sessions restore a
Qdrant snapshot from Drive in about 30 seconds.


## 0. Setup

In [ ]:
# COLAB ONLY — skip this cell if running on HPRC.
import sys, os

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Subprocesses (ingest_v4.py via !python) need this: sys.modules detection
# does NOT cross process boundaries, environment variables DO.
os.environ["MRAG_ENV"] = "colab"

REPO_URL = "https://github.com/hannanazad/Beyond_RAG.git"
REPO_DIR = "/content/Beyond_RAG"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

sys.path.insert(0, REPO_DIR)

# Step 1 — torch matched to Colab's CUDA 12.4.
# Colab ships torch 2.5.1, but transformers >=4.51 enforces CVE-2025-32434
# and refuses to load .bin checkpoints on torch <2.6. BGE-M3 ships .bin.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchvision==0.21.0

# Step 2 — everything else
!pip install -q -r $REPO_DIR/requirements.txt

# Step 3 — pip's bulk resolver does not enforce CEILINGS when an already
# installed version satisfies the lower bound. Colab ships newer packages
# than this pipeline can use, so pin them explicitly.
!pip install -q --no-deps --force-reinstall \
    "transformers>=4.49,<4.55" \
    "huggingface_hub>=0.34,<0.35" \
    "tokenizers>=0.21,<0.22" \
    "torchao>=0.13,<0.14"

!python -c "from transformers import PreTrainedModel; print('transformers import OK')"

In [ ]:
# API keys from Colab Secrets (key icon, left sidebar).
# Keys authenticate only; select the model with CFG.set_vlm_model(...).
from google.colab import userdata
import os

def _load(env_name, *secret_names):
    for s in secret_names:
        try:
            os.environ[env_name] = userdata.get(s)
            print(f"{env_name}: loaded (secret {s!r})")
            return
        except Exception:
            continue
    print(f"{env_name}: no matching secret (skipped)")

_load("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY", "QWEN")
_load("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY")
_load("GEMINI_API_KEY",    "GEMINI_API_KEY")

## 1. Sanity check

In [ ]:
import os, sys
sys.path.insert(0, "/content/Beyond_RAG" if os.path.isdir("/content/Beyond_RAG") else ".")

from mrag.config import CFG

print("Environment :", CFG.environment)
print("Base dir    :", CFG.base_dir, "| exists:", CFG.base_dir.exists())
print("PDF path    :", CFG.pdf_path, "| exists:", CFG.pdf_path.exists())
print("Qdrant dir  :", CFG.qdrant_dir)
print("Cache dir   :", CFG.cache_dir)
print("HF cache    :", CFG.hf_home)
print("VLM provider:", CFG.vlm_provider)
print("VLM model   :", CFG.vlm_model_api if CFG.vlm_provider == "api" else CFG.vlm_model)
print("API key set :", bool(os.environ.get(CFG.api_key_env_var)))
print()
print("Sheets per figure :", CFG.max_sheets_per_figure)
print("Images per request:", CFG.max_images_total)

try:
    import torch
    print("GPU         :", torch.cuda.get_device_name(0))
except Exception as e:
    print("GPU         : none —", e)

assert CFG.pdf_path.exists(), (
    f"No PDF at {CFG.pdf_path} or anywhere in {CFG.base_dir}. "
    f"Upload the MUTCD PDF to {CFG.base_dir} first."
)

## 2. Build or restore the vector store

Restores a Drive snapshot if one exists and the figure crops are v4.
Otherwise runs a full ingest.


In [ ]:
import shutil, tarfile, json
from pathlib import Path
from qdrant_client import QdrantClient

local_qdrant     = CFG.qdrant_dir                  # /content/qdrant_db (fast local SSD)
drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"  # snapshot on Drive

def _qdrant_ok(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        c = QdrantClient(path=str(path))
        try:
            return CFG.coll_chunks in {col.name for col in c.get_collections().collections}
        finally:
            c.close()
    except Exception:
        return False

def _v4_done() -> bool:
    """True once v4 (caption-below) figure extraction has run."""
    try:
        with open(CFG.figures_jsonl) as f:
            first = json.loads(next(f))
        return str(first.get("extraction_method", "")).startswith("caption_below_v2")
    except Exception:
        return False

if _qdrant_ok(local_qdrant) and _v4_done():
    print(f"Qdrant populated at {local_qdrant} and figures are v4. Nothing to do.")

elif drive_qdrant_tar.exists() and _v4_done():
    print(f"Restoring snapshot from {drive_qdrant_tar} "
          f"({drive_qdrant_tar.stat().st_size/1e6:.0f} MB)...")
    shutil.rmtree(local_qdrant, ignore_errors=True)
    local_qdrant.mkdir(parents=True, exist_ok=True)
    with tarfile.open(drive_qdrant_tar, "r") as tar:
        tar.extractall(local_qdrant.parent)
    assert _qdrant_ok(local_qdrant), "Extracted but collection not found."
    print("Local Qdrant verified OK.")

else:
    if drive_qdrant_tar.exists() and not _v4_done():
        print("Snapshot on Drive is stale (figures are pre-v4) — ignoring it.")
    else:
        print("No usable store found. Running ingest.")
    print("Expect ~30-45 min on an A100. Run the snapshot cell afterwards.")
    !cd /content/Beyond_RAG && python scripts/ingest_v4.py
    assert _qdrant_ok(local_qdrant), "Ingest finished but Qdrant is empty — check the log."
    assert _v4_done(),               "Ingest ran but figures.jsonl is not v2 — check the log."
    print("Ingest verified OK.")

### 2.1 Snapshot Qdrant back to Drive

Run after any successful ingest so the next session restores in seconds.

In [ ]:
if CFG.environment == "colab":
    local_qdrant     = CFG.qdrant_dir
    drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"

    if local_qdrant.exists():
        print(f"Snapshotting {local_qdrant} -> {drive_qdrant_tar} ...")
        drive_qdrant_tar.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(drive_qdrant_tar, "w") as tar:
            tar.add(local_qdrant, arcname=local_qdrant.name)
        print(f"Done ({drive_qdrant_tar.stat().st_size/1e6:.0f} MB).")
    else:
        print("No local Qdrant to snapshot.")

### 2.2 Repair colliding chunk IDs

`chunk_id` is `section + rule_type + ordinal`, which is **not unique**: a
paragraph split across a page break re-fires the ordinal. 36 IDs collided
across 107 rows, and because Qdrant upserts by `chunk_id_to_int(chunk_id)`,
the later row silently overwrote the earlier one.

`parsing.py` now suffixes repeats, so a **fresh ingest** is already fixed.
This cell repairs an **existing cache** without re-embedding — the dense
vectors for all rows already exist in file order, so only the IDs change.

Report first, then re-run with `--apply`, then rebuild the graph and
re-upsert.


In [ ]:
# Report only — nothing is written.
!cd /content/Beyond_RAG && python scripts/repair_chunk_ids.py --cache "$(python -c 'from mrag.config import CFG; print(CFG.cache_dir)')"

# To actually rewrite chunks.jsonl (a .prerepair.bak is kept), uncomment:
# !cd /content/Beyond_RAG && python scripts/repair_chunk_ids.py --cache "$(python -c 'from mrag.config import CFG; print(CFG.cache_dir)')" --apply

## 3. Initialise the pipeline

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')

if CFG.vlm_provider == "api" and not os.environ.get(CFG.api_key_env_var):
    print(f"NOTE: no key in {CFG.api_key_env_var!r} for {CFG.vlm_model_api!r}. "
          f"Init will proceed; load the key or switch with CFG.set_vlm_model(...).")

from mrag.ask import init_pipeline
pipeline = init_pipeline()
print("VLM loaded :", pipeline.vlm.loaded_name if pipeline.vlm else "none")
print("KG         :", pipeline.kg.g.number_of_nodes(), "nodes,",
                      pipeline.kg.g.number_of_edges(), "edges")

## 4. Ask

In [ ]:
from mrag.ask import ask

_ = ask("What is required when installing a STOP sign at an all-way stop intersection?")

In [ ]:
_ = ask("Explain Figure 2B-1 and the plaques it shows", show_scores=True)

In [ ]:
# Router smoke test: a definitional question should retrieve NO figures.
# Check the debug output — figure_router.needs_figures should be False.
_ = ask("What does 'shall' mean in the MUTCD?", show_scores=True)

## 5. Verify the multi-sheet figure fix

A canonical MUTCD figure or table can span several sheets. Retrieval always
emitted every sheet in `image_paths`, but the prompt builder read the
singular `image_path` — so **only sheet 1 ever reached the model**.

116 of 553 canonical entities are multi-sheet, hiding 174 sheets. 34 of the
150 benchmark questions have gold evidence on one of them. Table 2B-1 has
8 sheets and is the gold evidence for TB008, TB009 and TB026.


In [ ]:
import json, collections
from pathlib import Path

figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
canon = collections.defaultdict(list)
for f in figs:
    canon[(f["kind"], f["canonical_id"])].append(f)

multi = {k: v for k, v in canon.items() if len(v) > 1}
print(f"canonical entities : {len(canon)}")
print(f"multi-sheet        : {len(multi)}")
print(f"sheets beyond the first: {sum(len(v) - 1 for v in multi.values())}")
print(f"widest: {sorted(((len(v), k) for k, v in multi.items()), reverse=True)[:5]}")
print()
print(f"cap in use: max_sheets_per_figure={CFG.max_sheets_per_figure}, "
      f"max_images_total={CFG.max_images_total}")
covered = sum(1 for v in canon.values() if len(v) <= CFG.max_sheets_per_figure)
print(f"entities fully shown under the cap: {covered}/{len(canon)}")

In [ ]:
# End to end: a question whose gold evidence is the 8-sheet Table 2B-1.
# Expect several images, each labelled "[sheet N of M]".
_ = ask("What are the required sign sizes in Table 2B-1 for a STOP sign?",
        show_scores=True)

## 6. Run the MUTCD-150 benchmark

150 questions against the current config. Pick the model first — every row
you intend to compare must use the **same** model.


In [ ]:
# The runner is a Python API, not a CLI.
import sys
sys.path.insert(0, "/content/Beyond_RAG/benchmarks/mutcd150/v1")

from mutcd_benchmark_runner import run_benchmark
from mrag.ask import ask
from mrag.config import CFG

BENCH = "/content/Beyond_RAG/benchmarks/mutcd150/v1/mutcd_benchmark_questions_v1.jsonl"
OUT   = CFG.base_dir / "benchmark_runs"

paths = run_benchmark(
    CFG=CFG,
    ask_fn=ask,
    questions_path=BENCH,
    output_root=OUT,
    run_id="beyond_rag_001",
    models=[{"alias": "fable", "selector": "fable", "provider": "anthropic"}],
    prompt_style="fewshot",
    # max_questions=5,          # smoke test first
    # question_ids=["TB008", "TB009", "TB026"],   # the 8-sheet Table 2B-1 questions
    resume=True,
)
for k, v in paths.items():
    print(f"{k:<24} {v}")

## 7. Ablations

Six configs live in `ablations/configs/`: router, VLM figure filter, graph
proximity, rule-type weight, hierarchy prior, reranker. Results land in
`ablations/results/`, which ships empty.


In [ ]:
!ls /content/Beyond_RAG/ablations/configs

# Retrieval-only ablations need no generation model and cost nothing in API calls.
# A3 (graph proximity) and A4 (rule-type weight) were built but never reported.
# !cd /content/Beyond_RAG && python ablations/run_ablation.py \
#     --ablation A3_no_graph --retrieval-only \
#     --output ablations/results/A3_no_graph.json

## 8. Debug retrieval without calling the VLM

In [ ]:
from mrag.retrieval import Retriever

res = pipeline.retriever.retrieve("STOP sign sizes at an all-way stop")

print(f"{len(res.chunks)} chunks\n")
for c in res.chunks:
    print(f"  {c.get('section_id'):<10} {c.get('content_type'):<9} "
          f"p.{c.get('page_printed'):<5} score={c.get('score', 0):.3f}")

print(f"\n{len(res.figures)} figures")
for f in res.figures:
    n = len(f.get("image_paths") or [])
    print(f"  {f.get('figure_id'):<16} sheets={n:<3} source={f.get('source','?')}")

## 9. Inspect the knowledge graph

In [ ]:
import collections
kg = pipeline.kg
print(kg.g.number_of_nodes(), "nodes,", kg.g.number_of_edges(), "edges")
print()
print("node kinds:")
for k, v in collections.Counter(d.get("kind", "?") for _, d in kg.g.nodes(data=True)).most_common():
    print(f"  {k:<14} {v}")
print()
print("edge labels:")
for k, v in collections.Counter(d.get("label", "?") for *_, d in kg.g.edges(data=True)).most_common():
    print(f"  {k:<20} {v}")

In [ ]:
# Sections cross-referenced by a given section (cites_section edges).
target = "4K.04"
refs = set()
for c in (json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()):
    if c["section_id"] == target:
        refs.update(c.get("section_refs") or [])
print(f"{target} cross-references: {sorted(refs)}")